# 40장 · LangSmith 추적 · 실패한 실행에 피드백과 평가 예시 남기기

LangSmith에서 실행 한 건을 run이라고 부릅니다. @traceable을 함수에 붙이면 추적을 켠 환경에서 그 함수의 입력·출력과 호출 관계를 기록할 수 있습니다.

## traceable을 적용한 함수 호출하기

`@traceable`은 함수를 추적 대상으로 표시합니다. `trim`은 그대로 문자열을 반환합니다. 첫 셀은 `tracing_context(enabled=False)` 안에서 결과만 읽으므로 외부에 실행 기록을 보내지 않습니다. 새 커널에는 `%pip install langsmith`로 패키지를 준비하세요.

In [ ]:
from langsmith import traceable, tracing_context

@traceable(name="앞뒤 공백 제거")
def trim(text):
    return text.strip()

with tracing_context(enabled=False):
    print(trim("  사과  "))

사과


출력은 “사과”입니다. 여기서는 함수의 반환부터 확인하려고 이 호출의 추적을 껐습니다. 이어지는 실습에서 추적을 켜고 실행 한 건의 ID로 원격 피드백을 연결합니다.

## 응용 실습 · 실패한 실행에 피드백 달기

### 실행 기록에서 첫 후보와 최종 인용 구별하기

39장의 질문을 다시 열어 첫 검색이 보낸 쪽과 최종 답이 인용한 쪽을 따로 읽습니다. 경로 연습 파일이면 그 사실도 기록합니다. 뒤의 B형 오인 사례는 실패 지점을 찾는 별도 예입니다. 통합 ZIP을 모두 풀고 실행하세요. 내 파일이 없으면 출력에 표시된 교재 예시로 시작합니다.

In [ ]:
from pathlib import Path
import json
import sys

root = next(folder for folder in (Path.cwd(), *Path.cwd().parents)
            if (folder / "pre-mid/documents/workflow").is_dir())
sys.path.insert(0, str(root / "pre-mid/documents/workflow"))
from workbook_files import read_previous, save_record

handoff = root / "chapter39-handoff.json"
if handoff.is_file():
    trace39 = json.loads(handoff.read_text(encoding="utf-8"))
    print("내 작업 ·", handoff.name)
else:
    trace39 = read_previous(root, 39)
print(trace39["evidence_mode"], trace39["question"])
print("첫 후보", trace39["searches"][0]["ids"])
print("최종 인용", [p["id"] for p in trace39["citations"]])

내 작업 · chapter39-handoff.json
경로 연습 2027년 4월 15일 기준으로 A형의 파일이 나뉘는 간격은 얼마이고 그때 녹음도 멈추나요?
첫 후보 ['recorder-a-1.1.pdf#p2', 'recorder-a-1.0.pdf#p2', 'recorder-a-1.0.pdf#p1', 'recorder-a-1.1.pdf#p1']
최종 인용 ['recorder-a-1.1.pdf#p2']


## 1. 제품명 한 글자가 답을 바꾸는 곳

`pre-mid/documents/sources` 안의 `recorder-a-1.1.pdf`와 `recorder-b-1.0.pdf`를 열고 각각 첫 쪽을 읽으세요. 두 설명서 모두 녹음 표시등을 다룹니다. A형은 녹음 중 빨간 표시등이 켜지고 끌 수 없습니다. B형은 **녹음을 시작하기 전**에 숨김을 선택할 수 있습니다. B형도 이미 녹음 중이라면 바로 바꿀 수 없습니다.

먼저 “숨길 수 있다”라는 짧은 문장 옆에 필요한 조건을 적으세요. 모델 이름, 설정 시점, 설명서 판본 중 하나가 사라지면 독자가 어떤 잘못된 행동을 할 수 있을까요? 답변의 말투보다 먼저 이 차이를 알아야 담당자가 평가 기준을 쓸 수 있습니다.

**준비물**. 전체 실습 ZIP을 풀고 Jupyter 커널을 그 환경으로 선택합니다. 앞 장에서 쓰던 패키지에 LangSmith SDK가 필요합니다. 설치가 필요한 커널에서는 별도 셀에 `%pip install "langsmith>=0.4,<1" langchain-openai langgraph chromadb pypdf python-dotenv streamlit`를 실행한 뒤 커널을 재시작하세요. 압축을 푼 폴더의 `.env`에 개인 실습용 `LANGSMITH_API_KEY`, `OPENAI_API_KEY`, 세 장에서 사용할 같은 `OPENAI_MODEL`을 둡니다. LangSmith 워크스페이스가 여럿이면 해당 워크스페이스의 키 또는 `LANGSMITH_WORKSPACE_ID`도 확인하세요. 팀이 선택한 LangSmith 엔드포인트가 있다면 `LANGSMITH_ENDPOINT`도 맞춥니다. 모델 호출은 계정 사용량에 반영됩니다. 원격 기록에는 제공된 가상 제품 자료를 사용합니다.

In [ ]:
from pathlib import Path
import json
import os
import sys
from dotenv import load_dotenv

# 전체 실습 묶음을 푼 폴더 또는 그 안의 장 폴더에서 노트북을 엽니다.
base = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pre-mid/documents/evaluation/eval_lab.py").is_file())
docs = base / "pre-mid/documents"
sys.path.insert(0, str(docs / "evaluation"))
load_dotenv(base / ".env")
from eval_lab import examples

# 키의 실제 값은 출력하지 않습니다. 개인 실습 공간의 키를 사용하세요.
smith_ready = bool(os.getenv("LANGSMITH_API_KEY") or os.getenv("LANGCHAIN_API_KEY"))
model_ready = bool(os.getenv("OPENAI_API_KEY"))
if smith_ready:
    os.environ["LANGSMITH_TRACING"] = "true"
    from langsmith import Client
    client = Client()
work = base / "my-recorder-evaluation"
work.mkdir(exist_ok=True)
print("LangSmith 연결 설정", smith_ready, "/ 모델 연결 설정", model_ready)

## 2. 검색이 틀렸나, 답을 쓸 때 틀렸나

**실행 기록(trace)** 은 질문을 받은 뒤 답을 돌려주기까지 일어난 일을 묶어 보여 줍니다. 그 안에서 검색이나 생성처럼 한 번 실행된 작업을 **run**이라고 합니다. 같은 문장을 두 번 보내도 실행 ID는 서로 다릅니다. ID가 있어야 어떤 사용자가 어느 답에 이의를 제기했는지 연결할 수 있습니다.

두 가지 가능성을 구분하세요. 검색 결과에 B형 1쪽만 있었다면 A형에 대한 근거가 모델에 도착하지 않았습니다. 반면 두 제품의 올바른 문단이 다 있었는데 답이 B형 설정만 가리킨다면, 근거를 고르는 단계나 최종 답을 쓰는 지시부터 봅니다. 뒤의 답이 틀렸다는 사실만으로 앞의 검색 설정을 바꿀 수는 없습니다.

`@traceable(name=...)`은 바로 아래 함수의 호출을 추적할 수 있게 합니다. 외부 추적을 켜고 인증하면 바깥 `replay` 한 건 아래에 내부 `retrieve_demo` 호출이 보입니다. `langsmith_extra`는 추적용 설정입니다. 이름과 ID를 붙여 어떤 실행에 피드백을 달지 정합니다. 반환값의 `source`가 B형이면 모델 문장을 고치기 전부터 이미 질문과 다른 자료가 선택된 것입니다.

In [2]:
from pypdf import PdfReader
from langsmith import traceable

@traceable(name="잘못 연결한 검색")
def retrieve_demo(question):
    # 장애 재현용입니다. 질문에 관계없이 B형 1쪽을 넘기는 실수를 심어 두었습니다.
    name = "recorder-b-1.0.pdf"
    page = PdfReader(docs / "sources" / name).pages[0].extract_text()
    return {"source": name, "page": 1, "text": page}

@traceable(name="장애 재현 답변")
def replay(question):
    hit = retrieve_demo(question)
    # 상담 화면에서 본 오류를 재현한 고정 문구입니다. 실시간 모델 출력과 구별합니다.
    return {"answer": "녹음 전 표시등을 숨김으로 설정하세요.",
            "source": hit["source"], "page": hit["page"]}

from uuid import uuid4
question = examples("dev")[0]["inputs"]["question"]
run_id = uuid4()
project_name = "recorder-classroom"
result = replay(question, langsmith_extra={"run_id": run_id, "project_name": project_name})
print(result["answer"], result["source"], result["page"])
if smith_ready:
    print("Tracing에서 열 프로젝트와 최상위 실행 ID", project_name, run_id)

녹음 전 표시등을 숨김으로 설정하세요. recorder-b-1.0.pdf 1


출력의 파일명이 B형입니다. 파일명이 실제로 존재하는지와 그 파일이 질문에 맞는지는 별개의 판단입니다. LangSmith를 연결했다면 Tracing에서 `recorder-classroom` 프로젝트를 열고 방금 표시한 ID와 같은 실행을 찾으세요. 바깥쪽 ‘장애 재현 답변’의 Inputs에서 A형을, 안쪽 ‘잘못 연결한 검색’의 Outputs에서 B형을 찾습니다. 이 실행에서 처음 어긋난 자리는 검색입니다. 원격 키가 없다면 위 셀의 로컬 결과로 문서를 대조할 수 있고, 온라인 실행 ID는 생성 대상으로 기록하지 않습니다.

## 3. create_feedback으로 한 실행에 점수 달기

`client.create_feedback`에서 `run_id`는 평가할 실행 한 건, `key`는 평가 항목, `score=0`은 이 실행의 기종과 인용이 어긋났다는 점수입니다. `comment`에는 사람이 PDF로 확인할 근거를 적습니다. 이 호출은 실행에 평가를 남깁니다. 다음 실험에서 같은 질문을 반복하려면 뒤에서 질문과 기대 답을 평가 데이터셋에 따로 담아야 합니다.

“별로예요”만 남기면 무엇을 고칠지 알 수 없습니다. **질문한 기기, 실제 인용, 올바른 설명서 쪽과 조건**을 한 줄에 담으세요. 아래 점수 0은 이 재현 사례에서 ‘인용한 기종이 맞는가’라는 질문에 대한 답입니다. 정상 실행에 무조건 0을 붙이면 평가 이력까지 잘못됩니다.

In [ ]:
from uuid import uuid5
if smith_ready:
    # 같은 화면에서 버튼을 다시 눌러도 같은 피드백 ID를 사용합니다.
    feedback_id = uuid5(run_id, "citation-device")
    client.create_feedback(run_id=run_id, key="citation_device_match", score=0,
        comment="질문은 A형, 인용은 B형 1.0판 1쪽. A형 1.1판 1쪽은 녹음 중 표시등을 끌 수 없다고 한다.",
        feedback_id=feedback_id)
    print("실행의 Feedback 탭에서 확인", run_id)

39장의 실제 챗봇에서는 답을 화면에 출력할 때 최상위 실행 ID도 그 답과 함께 세션에 저장합니다. 사용자가 이전 답에 평가를 남겼다면 **화면에서 선택한 답의 ID**가 필요합니다. 아래는 수업에서 직접 띄우는 작은 피드백 화면입니다. 위에서 출력한 ID를 넣으면 원격에 저장된 질문과 답을 먼저 보여 줍니다. 실제 39장 실행의 ID도 같은 워크스페이스에 저장했다면 사용할 수 있습니다.

In [4]:
from textwrap import dedent
# 노트북에서 실행 파일을 준비합니다. 학생이 별도 소스 파일을 옮길 필요가 없습니다.
app = dedent("""
    from pathlib import Path
    from uuid import UUID, uuid5
    import streamlit as st
    from dotenv import load_dotenv
    from langsmith import Client
    load_dotenv(Path(__file__).resolve().parents[1] / ".env")
    st.title("녹음기 답변 피드백")
    trace_id = st.text_input("선택한 답의 실행 ID")
    if trace_id:
        client = Client()
        run = client.read_run(UUID(trace_id))
        st.write(run.inputs, run.outputs)
        with st.form("review"):
            matched = st.radio("인용한 문서의 기종이 맞나요?", ["맞음", "틀림"])
            note = st.text_input("질문한 기종, 실제 인용, 확인한 쪽")
            send = st.form_submit_button("이 실행에 피드백 저장")
        if send and note.strip():
            # 수업에서는 본인의 계정에서 실행 하나당 한 개의 판단을 남깁니다.
            client.create_feedback(run_id=run.id, key="device_review", score=int(matched == "맞음"),
                comment=note, feedback_id=uuid5(run.id, "classroom-device-review"))
            st.success("저장했습니다.")
""")
(work / "review_app.py").write_text(app, encoding="utf-8")
print("압축을 푼 폴더의 터미널에서 실행: python -m streamlit run my-recorder-evaluation/review_app.py")

압축을 푼 폴더의 터미널에서 실행: python -m streamlit run my-recorder-evaluation/review_app.py


## 실습 · 같은 오답을 낸 두 팀에게 서로 다른 수정 제안하기

교사에게 받은 아래 두 기록은 진단 연습을 위해 구성한 예시입니다. 두 팀 모두 A형 사용자에게 “녹음 전에 숨김을 설정하세요”라고 안내했습니다.

| 팀 | 검색 출력 | 근거 판단이 넘긴 쪽 | 답의 인용 |
|---|---|---|---|
| 보라 | B형 1.0판 1쪽 | B형 1.0판 1쪽 | B형 1.0판 1쪽 |
| 초록 | A형 1.1판 1쪽, B형 1.0판 1쪽 | B형 1.0판 1쪽 | B형 1.0판 1쪽 |

인용 표시만 바꾸겠다는 제안을 받아들여도 될까요? 각 팀에서 **처음 바로잡을 출력 하나**와 **그 수정이 효과가 있었는지 확인할 질문**을 적으세요. 팀끼리 바꾸어 읽고 그 수정으로는 고치기 어려운 실패가 무엇인지 물어보세요.

In [ ]:
# 보라 팀과 초록 팀에서 직접 확인할 첫 번째 값을 작성합니다.
first_check = {}

<details class="notebook-answer"><summary>예시 풀이 열기</summary><pre><code>first_check = {&quot;보라&quot;: &quot;첫 검색 후보에 A형 1.1판 1쪽이 있는가&quot;,
               &quot;초록&quot;: &quot;근거 판단 뒤에 선택된 쪽의 모델명이 A인가&quot;}
print(first_check)</code></pre><p>보라 팀은 A형 쪽이 후보에 들어오는지부터 봅니다. 초록 팀에서는 A형 쪽이 이미 들어왔으므로 근거 판단이 선택된 기종과 판본을 지키는지 먼저 봅니다. 올바른 문서명만 붙이고 “숨김을 설정하라”를 그대로 두면 사용자는 여전히 잘못 안내받습니다. D02처럼 B형의 숨김 설정이 실제로 가능한 질문도 함께 시험해야 모든 답을 A형으로 고친 실수를 발견합니다.</p></details>

## examples로 평가 입력과 참고 답을 따로 읽기

`examples("dev")`는 수업용 개발 질문과 참고 답을 한 건씩 짝지어 리스트로 반환합니다. `row["inputs"]`는 챗봇에 보낼 값이고 `row["outputs"]`는 실행이 끝난 뒤 대조할 값입니다. 정답 쪽수가 챗봇 입력 안에 있으면 검색 능력을 시험할 수 없습니다.

**데이터셋**은 같은 시험을 다시 볼 수 있도록 입력과 판단 기준을 짝지은 묶음입니다. 질문에는 사용자가 아는 조건을 둡니다. 참고 답에는 담당자가 PDF를 보고 판정한 쪽과 의미를 둡니다. 올바른 페이지를 모델의 질문에 넣으면 검색 실패가 사라져 버립니다.

개발용 CSV의 D03과 D04는 같은 A형의 파일 분할 간격을 묻습니다. D03은 3월이므로 1.0판 2쪽, D04는 4월이므로 1.1판 2쪽입니다. D04에서 ‘45분’만 맞고 녹음이 멈춘다고 했다면 아직 실패입니다. D05의 무선 전송 방법은 설명서로 확인할 수 없으므로 기대 행동을 보류로 적고 정답 페이지를 비웁니다. ‘기능이 없다’는 단정도 문서로 뒷받침되지 않습니다.

### 같은 실행의 같은 항목을 식별하기

In [ ]:
from uuid import UUID, uuid5

sample_run = UUID("00000000-0000-0000-0000-000000000001")
first_id = uuid5(sample_run, "citation-device")
print(first_id == uuid5(sample_run, "citation-device"))
print(first_id == uuid5(sample_run, "page-match"))

True
False


첫 결과가 True라서 같은 실행에서 같은 항목을 다시 선택하면 같은 피드백 ID를 계산합니다. 둘째 결과가 False인 것은 실행이 같아도 평가 항목 이름이 달라졌기 때문입니다. 이 셀은 원격 피드백을 등록하지 않습니다.

In [6]:
dev = examples("dev")
# inputs는 챗봇이 보는 값, outputs는 나중에 평가자가 보는 참고 답입니다.
print(dev[3]["inputs"])
print(dev[3]["outputs"])

{'id': 'D04', 'question': '2027년 4월 15일 기준으로 A형의 파일이 나뉘는 간격은 얼마이고 그때 녹음도 멈추나요?', 'model': 'A', 'edition': '1.1'}
{'action': 'answer', 'source': 'recorder-a-1.1.pdf', 'page': 2, 'reference_answer': '45분마다 파일이 나뉘고 녹음은 계속된다.'}


도움 자료 `evaluation/eval_lab.py`가 같은 질문 ID의 두 CSV 행을 연결합니다. `model`은 제품 A/B, `edition`은 챗봇 화면에서 선택한 판본입니다. API의 언어 모델 이름과 혼동하지 않도록 실험 설정에는 언어 모델 이름을 따로 남깁니다.

In [ ]:
from eval_lab import dataset_for
if smith_ready:
    dataset = dataset_for(client, "dev", dev)
    print("Datasets에서 찾을 이름과 ID", dataset.name, dataset.id)
    print("저장한 질문 수", len(list(client.list_examples(dataset_id=dataset.id))))

이름 뒤의 짧은 문자열은 질문과 참고 답의 내용으로 만들었습니다. 같은 내용은 같은 세트를 열고 행이 달라지면 새 세트를 엽니다. Datasets에서 D04의 Inputs와 Reference Outputs를 열고 위 출력과 대조하세요. 마지막 확인용 H01~H04는 개발 단계에서 여는 대신, 42장에 가서 선택한 설정을 검사할 때 사용합니다.

## 개인 실습 · 담당자 두 명의 오류 진단서

두 사람 모두 D01의 원문을 읽습니다. 한 사람은 이 장의 재현 실행을 조사하고, 다른 사람은 39장에서 완성한 실제 그래프에 D01을 보냅니다. 둘 다 **질문 → 첫 검색의 후보 → 근거 판단 → 최종 인용 → 사용자 피드백** 순으로 기록합니다. 실제 그래프가 처음부터 맞게 답했다면 정상 실행이 어떤 증거로 오류를 피했는지 적으세요. 이어서 D03과 D04의 기대 답을 정답표 없이 먼저 작성하고, 동료와 판본·쪽·필수 조건을 대조하세요.

제출물은 실제로 열 수 있는 실행 ID와 피드백, 두 질문의 입력·참고 답, 수정이 필요한 첫 단계 한 줄입니다. 공개된 실습 자료만 포함하세요. 활동지 `pre-mid/documents/evaluation/fieldbook.md`의 40장 칸을 이용합니다.

In [ ]:
# D03의 기종·판본·쪽·답을 직접 적은 다음, 제공된 대조표와 비교하세요.
my_reference = {}
# 쪽을 특정할 source와 page, 기대 행동 action, 그 쪽에서 확인할 reference_answer를 써 보세요.

<details class="notebook-answer"><summary>모범 예시와 판정 근거 보기</summary><pre><code>my_reference = {&quot;source&quot;: &quot;recorder-a-1.0.pdf&quot;, &quot;page&quot;: 2, &quot;action&quot;: &quot;answer&quot;, &quot;reference_answer&quot;: &quot;30분마다 새 파일이 열립니다.&quot;}
print(my_reference)</code></pre><p>재현 실행에서 기대한 기종은 A이고 검색 출력의 source는 B입니다. 처음 어긋난 곳은 검색입니다. D03은 3월에 30분마다 새 파일이 열립니다. D04는 4월부터 45분마다 나뉘고 녹음은 계속된다는 두 조건을 함께 채점합니다. 실제 그래프의 실행 ID와 평가는 본인 결과를 기록하세요.</p></details>

## 실행 기록에 사람의 참고 답 붙이기

D04의 원문 쪽을 직접 읽고, 행동·해당 쪽·참고 답을 평가용으로만 붙이세요. 자신이 질문을 바꿨다면 다음 셀의 reference도 고칩니다. 저장한 사례는 다음 장에서 그대로 채점합니다.

In [ ]:
reference = {"action": "answer", "source": "recorder-a-1.1.pdf", "page": 2,
             "reference_answer": "45분마다 파일이 나뉘고 녹음은 계속된다."}
example = {"inputs": {"id": "MY01", "question": trace39["question"],
                      "model": trace39["model"], "edition": trace39["edition"]},
           "outputs": reference, "observed": trace39}
print("평가자만 읽는 쪽", reference["source"], reference["page"])
save_record(root, 40, example)

평가자만 읽는 쪽 recorder-a-1.1.pdf 2
my-course/workflow/chapter40.json


**다음 장에 가져갈 것**. 개인 개발 세트의 이름과 버전, 39장과 24장의 색인 버전, 잘못을 발견한 질문 ID를 보관합니다. 41장에서는 같은 질문을 다시 사용해 구현 변경이 효과가 있었는지 비교합니다.

[LangSmith에서 실행에 피드백 붙이기](https://docs.langchain.com/langsmith/attach-user-feedback) · [데이터셋 작성과 버전 관리](https://docs.langchain.com/langsmith/manage-datasets-programmatically)